# Taller Interactivo: Conversión de Unidades y Cinemática 1D (MRU/MRUA)
### Física BR · Física Mecánica

Este taller cubre la **conversión de unidades** entre sistemas imperial y SI, y el **análisis de movimiento en una dimensión** (MRU y MRUA), con énfasis en la interpretación de gráficas v-t y la diferencia entre desplazamiento y distancia recorrida.

**Contenido:**
- **Punto 1:** Conversión de unidades imperiales a SI (tubería de acero)
- **Punto 2:** Análisis gráfico del movimiento — aceleración, desplazamiento y distancia
- **Punto 3:** Cinemática aplicada al tejo colombiano

Cada punto incluye un esquema, controles interactivos, solución paso a paso y un cuestionario de autoevaluación (nota formativa de 0,0 a 5,0).

---


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))



try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass


## Punto 1 — Conversión de unidades: tubería de acero  ★  [CL, UFT]
Un proveedor internacional envía las especificaciones de una tubería de acero en **unidades imperiales**, y el ingeniero debe convertirlas al **Sistema Internacional (SI)** para cargarlas en la bitácora de cálculo.

**Especificaciones originales:** diámetro 2,00 in · longitud 20,0 ft · masa lineal 3,50 lb/ft · presión de prueba 150 psi.

**Factores de conversión:** 1,00 in = 0,0254 m · 1,00 ft = 0,305 m · 1,00 lb = 0,454 kg · 1,00 psi = 6,89 kPa.

Explora cómo cambian los valores en SI al variar las especificaciones originales.

> 🔧 **Aplicación en ingeniería:** un error en la conversión entre sistemas imperial y métrico puede causar desastres reales. El orbitador Mars Climate (NASA, 1999) se perdió por una confusión libras-newtons; en obra, un error de unidades en la presión de prueba de una tubería puede llevar a fallas estructurales o sobrecostos millonarios.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,ax=plt.subplots(figsize=(8,3.5))
    # tubería (cilindro visto de lado)
    ax.add_patch(plt.Rectangle((1,1.5),5,1,color='#7f8c8d',alpha=0.5,ec='#2c3e50',lw=1.5))
    ax.add_patch(plt.Rectangle((1,1.5),5,1,fill=False,ec='#2c3e50',lw=1.5))
    from matplotlib.patches import Ellipse
    ax.add_patch(Ellipse((1,2),0.4,1,color='#95a5a6',ec='#2c3e50',lw=1.5))
    ax.add_patch(Ellipse((6,2),0.4,1,fill=False,ec='#2c3e50',lw=1.5))
    # cotas
    ax.annotate('',xy=(6,3),xytext=(1,3),arrowprops=dict(arrowstyle='<->',color='#2980b9',lw=1.5))
    ax.text(3.5,3.2,'longitud (20,0 ft → SI)',ha='center',fontsize=9,color='#2980b9',fontweight='bold')
    ax.annotate('',xy=(0.6,1.5),xytext=(0.6,2.5),arrowprops=dict(arrowstyle='<->',color='#c0392b',lw=1.5))
    ax.text(0.3,2,'diámetro\n(2,00 in)',ha='right',va='center',fontsize=8,color='#c0392b',fontweight='bold')
    # etiquetas de conversión
    ax.text(3.5,0.7,'imperial → Sistema Internacional (SI)',ha='center',fontsize=10,color='#2c3e50',fontweight='bold',
            bbox=dict(boxstyle='round',fc='#fdf6e3',ec='#d9a441'))
    ax.set_xlim(0,7.5); ax.set_ylim(0,4); ax.axis('off')
    ax.set_title('Tubería de acero: especificaciones a convertir')
    plt.tight_layout(); plt.show()
esquema()

def conversion(diam_in=2.00, long_ft=20.0, masa_lbft=3.50, pres_psi=150.0):
    diam_m=diam_in*0.0254
    long_m=long_ft*0.305
    masa_kgm=masa_lbft*0.454/0.305
    pres_kpa=pres_psi*6.89
    fig,ax=plt.subplots(figsize=(9,4))
    specs=['Diámetro\n(m)','Longitud\n(m)','Masa lineal\n(kg/m)','Presión\n(kPa)']
    vals=[diam_m, long_m, masa_kgm, pres_kpa]
    # normalizar para verlos en la misma escala (log)
    import numpy as _np
    colores=['#c0392b','#2980b9','#e67e22','#27ae60']
    y=_np.arange(len(specs))
    for i,(s,v,c) in enumerate(zip(specs,vals,colores)):
        ax.barh(i, _np.log10(v+1), color=c, alpha=0.7)
        ax.text(_np.log10(v+1)+0.05, i, f'{v:.3g}', va='center', fontsize=11, fontweight='bold')
    ax.set_yticks(y); ax.set_yticklabels(specs)
    ax.set_xlabel('escala logarítmica (los valores reales están anotados)')
    ax.set_title('Especificaciones convertidas al SI')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

interact(conversion,
    diam_in=FloatSlider(value=2.00,min=1,max=6,step=0.5,description='diám (in)'),
    long_ft=FloatSlider(value=20.0,min=10,max=40,step=5,description='long (ft)'),
    masa_lbft=FloatSlider(value=3.50,min=1,max=8,step=0.5,description='masa (lb/ft)'),
    pres_psi=FloatSlider(value=150.0,min=50,max=300,step=25,description='presión (psi)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                ee=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+ee
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    _pc("a) Diámetro:  2,00 in × 0,0254 m/in = "+f"{2.00*0.0254:.4g} m")
    _pc("   Longitud:  20,0 ft × 0,305 m/ft = "+f"{20.0*0.305:.3g} m")
    _pc("b) Masa lineal:  3,50 lb/ft × (0,454 kg/lb) / (0,305 m/ft) = "+f"{3.50*0.454/0.305:.3g} kg/m")
    _pc("c) Presión:  150 psi × 6,89 kPa/psi = "+f"{150*6.89:.4g} kPa")
    _pc("")
    _pc("d) Un error de unidades puede ser catastrófico: confundir psi con kPa daría una presión")
    _pc("   de prueba ~7 veces distinta, llevando a una tubería sub-dimensionada que falla, o")
    _pc("   sobre-dimensionada que dispara los costos. Ejemplo real: el Mars Climate Orbiter (1999).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Convierta el diámetro de la tubería (2,00 in) a metros.",0.0508,0.03,"m",
    "2,00 in × 0,0254 m/in = 0,0508 m.", id_preg="Cin_p1_1", peso=1.0)
quiz_numerico("Convierta la masa lineal (3,50 lb/ft) a kg/m.",5.21,0.03,"kg/m",
    "3,50 lb/ft × 0,454 kg/lb ÷ 0,305 m/ft ≈ 5,21 kg/m.", id_preg="Cin_p1_2", peso=1.0)
quiz_numerico("Convierta la presión de prueba (150 psi) a kPa.",1034,0.03,"kPa",
    "150 psi × 6,89 kPa/psi = 1034 kPa ≈ 1,03 MPa.", id_preg="Cin_p1_3", peso=1.0)
quiz_opcion_multiple("¿Por qué es crítico convertir correctamente entre sistemas de unidades en ingeniería?",
    ["No es importante, los números son parecidos","Un error puede causar fallas estructurales, sobrecostos o pérdidas totales (como el Mars Climate Orbiter)","Solo importa en física teórica","Las unidades imperiales son más precisas"],1,
    "Un error de conversión llevó a la pérdida del Mars Climate Orbiter (NASA, 1999). En obra puede causar fallas o sobrecostos graves.", id_preg="Cin_p1_4", peso=1.0)


## Punto 2 — Análisis gráfico del movimiento (MRU/MRUA)  ★★★  [UCM, PRC, ULC]
Un prototipo de vehículo se desplaza en línea recta. El sensor a bordo registra la velocidad v(t) durante 10,0 s. En t=0, parte de x₀=0,00 m con velocidad de 12,0 m/s. La gráfica v(t) baja de +12,0 m/s a 0 en t=4,00 s, y sigue bajando hasta −12,0 m/s en t=10,0 s.

**Conceptos clave:**
- La **aceleración** es la pendiente de la gráfica v-t.
- El **desplazamiento** es el área bajo la curva v-t (con signo).
- La **distancia total** suma las áreas en valor absoluto (sin importar el signo).

Este punto muestra la diferencia fundamental entre desplazamiento y distancia recorrida.

> 🔧 **Aplicación en ingeniería:** interpretar gráficas v-t es esencial en sistemas de control de vehículos, análisis de sensores inerciales y telemetría. Un sistema de frenado regenerativo (como en el Metro de Medellín) necesita distinguir cuándo el vehículo desacelera y cuándo invierte su marcha.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,ax=plt.subplots(figsize=(8.5,4.5))
    # gráfica v-t del enunciado
    t=np.array([0,4,10]); v=np.array([12,0,-12])
    ax.plot(t,v,color='#2980b9',lw=3,zorder=3)
    ax.scatter(t,v,s=80,color='#2980b9',zorder=5,edgecolor='white',linewidth=1.5)
    ax.axhline(0,color='#2c3e50',lw=1.5)
    # área positiva (desplazamiento +) y negativa
    t1=np.linspace(0,4,50); v1=12-3*t1
    t2=np.linspace(4,10,50); v2=-2*(t2-4)
    ax.fill_between(t1,0,v1,color='#27ae60',alpha=0.2,label='área + (avanza): +24 m')
    ax.fill_between(t2,0,v2,color='#e74c3c',alpha=0.2,label='área − (retrocede): −36 m')
    # anotar pendientes
    ax.annotate('pendiente a₁=−3,00 m/s²',xy=(2,6),xytext=(4.5,8),fontsize=8,color='#8e44ad',
                arrowprops=dict(arrowstyle='->',color='#8e44ad'))
    ax.annotate('pendiente a₂=−2,00 m/s²',xy=(7,-6),xytext=(2,-10),fontsize=8,color='#8e44ad',
                arrowprops=dict(arrowstyle='->',color='#8e44ad'))
    ax.set_xlabel('tiempo t (s)'); ax.set_ylabel('velocidad v (m/s)')
    ax.set_title('Gráfica v(t): la pendiente es la aceleración, el área es el desplazamiento')
    ax.legend(fontsize=8,loc='upper right'); ax.set_ylim(-14,14); ax.grid(alpha=0.25,linestyle='--')
    plt.tight_layout(); plt.show()
esquema()

def analisis(v0=12.0, t_cero=4.0, v_final=-12.0, t_total=10.0):
    # dos tramos: [0,t_cero] de v0 a 0; [t_cero,t_total] de 0 a v_final
    a1=(0-v0)/t_cero
    a2=(v_final-0)/(t_total-t_cero)
    # áreas
    A1=0.5*t_cero*v0
    A2=0.5*(t_total-t_cero)*v_final
    desplaz=A1+A2
    dist=abs(A1)+abs(A2)
    t=np.linspace(0,t_total,300)
    v=np.where(t<=t_cero, v0+a1*t, a2*(t-t_cero))
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4))
    # v-t
    ax1.plot(t,v,color='#2980b9',lw=2.5)
    ax1.fill_between(t,0,v,where=(v>=0),color='#27ae60',alpha=0.15)
    ax1.fill_between(t,0,v,where=(v<0),color='#e74c3c',alpha=0.15)
    ax1.axhline(0,color='#2c3e50',lw=1)
    ax1.set_xlabel('t (s)'); ax1.set_ylabel('v (m/s)'); ax1.set_title('Velocidad v(t)'); ax1.grid(alpha=.3)
    # x-t (posición, integral)
    x=np.zeros_like(t)
    for i in range(1,len(t)):
        x[i]=x[i-1]+0.5*(v[i]+v[i-1])*(t[i]-t[i-1])
    ax2.plot(t,x,color='#8e44ad',lw=2.5)
    ax2.axhline(0,color='#2c3e50',lw=1)
    ax2.scatter([t_total],[x[-1]],color='#c0392b',s=70,zorder=5,label=f'x final={x[-1]:.3g} m')
    ax2.set_xlabel('t (s)'); ax2.set_ylabel('posición x (m)'); ax2.set_title('Posición x(t)'); ax2.legend(fontsize=8); ax2.grid(alpha=.3)
    plt.suptitle(f'Desplazamiento={desplaz:.3g} m  ·  Distancia total={dist:.3g} m  (¡son distintos!)',
                 fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()

interact(analisis,
    v0=FloatSlider(value=12.0,min=6,max=18,step=2,description='v₀ (m/s)'),
    t_cero=FloatSlider(value=4.0,min=2,max=6,step=1,description='t(v=0) (s)'),
    v_final=FloatSlider(value=-12.0,min=-18,max=-6,step=2,description='v final (m/s)'),
    t_total=FloatSlider(value=10.0,min=8,max=12,step=1,description='t total (s)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                ee=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+ee
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    _pc("a) ACELERACIÓN por tramos (pendiente de v-t):")
    _pc("   Tramo 1 [0; 4,00 s]:  a₁ = (0−12,0)/(4,00−0) = −3,00 m/s²")
    _pc("   Tramo 2 [4,00; 10,0 s]:  a₂ = (−12,0−0)/(10,0−4,00) = −2,00 m/s²")
    _pc("   Interpretación: en el tramo 1 el vehículo FRENA (v>0 pero a<0, va parando).")
    _pc("   En el tramo 2 se mueve en SENTIDO CONTRARIO (v<0), acelerando hacia atrás.")
    _pc("")
    _pc("c) DESPLAZAMIENTO (área bajo v-t, con signo):")
    _pc("   Área tramo 1 = ½·4,00·12,0 = +24,0 m")
    _pc("   Área tramo 2 = ½·6,00·(−12,0) = −36,0 m")
    _pc("   Desplazamiento = +24,0 + (−36,0) = −12,0 m  → x(10,0 s) = −12,0 m")
    _pc("")
    _pc("d) DISTANCIA TOTAL (áreas en valor absoluto):")
    _pc("   Distancia = |+24,0| + |−36,0| = 60,0 m")
    _pc("   La distancia (60,0 m) ≠ el desplazamiento (−12,0 m) porque el vehículo")
    _pc("   avanzó 24 m, se detuvo y luego retrocedió 36 m: quedó 12 m DETRÁS del origen,")
    _pc("   pero su recorrido total fue de 60 m.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("En la gráfica v(t), ¿cuál es la aceleración en el tramo [0; 4,00 s]? (en m/s²)",-3.00,0.03,"m/s²",
    "a₁=(0−12,0)/(4,00−0)=−3,00 m/s². Negativa: el vehículo frena.", id_preg="Cin_p2_1", peso=3.0)
quiz_numerico("¿Cuál es la aceleración en el tramo [4,00; 10,0 s]? (en m/s²)",-2.00,0.03,"m/s²",
    "a₂=(−12,0−0)/(10,0−4,00)=−2,00 m/s².", id_preg="Cin_p2_2", peso=3.0)
quiz_numerico("Usando el área bajo v-t, ¿cuál es el desplazamiento total (posición final x en t=10,0 s)? (en m)",-12.0,0.05,"m",
    "Área=½·4·12 + ½·6·(−12)=+24−36=−12,0 m. El vehículo queda 12 m detrás del origen.", id_preg="Cin_p2_3", peso=3.0)
quiz_numerico("¿Cuál es la distancia total recorrida entre t=0 y t=10,0 s? (en m)",60.0,0.03,"m",
    "Distancia=|+24,0|+|−36,0|=60,0 m (suma de áreas en valor absoluto).", id_preg="Cin_p2_4", peso=3.0)
quiz_opcion_multiple("¿Por qué el desplazamiento (−12,0 m) y la distancia (60,0 m) son distintos?",
    ["Por un error de cálculo","Porque el vehículo avanzó y luego retrocedió: el desplazamiento mide la posición neta y la distancia el recorrido total","Porque la velocidad es negativa","Siempre son iguales"],1,
    "El desplazamiento es la posición neta (con signo); la distancia es todo el camino recorrido. Como el vehículo retrocedió, difieren.", id_preg="Cin_p2_5", peso=3.0)


## Punto 3 — Cinemática aplicada al tejo colombiano  ★★  [UCM, PRC]
En una partida de **tejo** (deporte tradicional colombiano), un jugador acelera el disco metálico uniformemente desde el reposo (v₀=0) a lo largo de una distancia de impulso d=1,20 m antes de soltarlo.

**Ecuaciones de MRUA** (aceleración constante):
- v² = v₀² + 2·a·d  (velocidad-distancia, sin tiempo)
- v = v₀ + a·t  (velocidad-tiempo)

Explora cómo la aceleración de impulso determina la velocidad de despegue del tejo.

> 🔧 **Aplicación en ingeniería:** el mismo principio del impulso en una distancia fija aparece en lanzadores, catapultas industriales, sistemas de eyección y cañones de riel. La relación v²=2·a·d permite dimensionar la aceleración necesaria para alcanzar una velocidad objetivo en un espacio limitado.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,ax=plt.subplots(figsize=(8,3.8))
    # cancha de tejo: fase de impulso + vuelo
    ax.axhspan(0,0.3,color='#d9c8a3',alpha=0.5)  # suelo
    # zona de impulso (mano acelera el tejo)
    ax.annotate('',xy=(3,1),xytext=(0.5,1),arrowprops=dict(arrowstyle='-|>',color='#e67e22',lw=2.5))
    ax.text(1.75,1.4,'fase de impulso\n(acelera en d=1,20 m)',ha='center',fontsize=9,color='#e67e22',fontweight='bold')
    # tejo (disco)
    from matplotlib.patches import Ellipse
    ax.add_patch(Ellipse((3.2,1),0.5,0.25,color='#7f8c8d',ec='#2c3e50',lw=1.5,zorder=5))
    ax.text(3.2,0.55,'tejo',ha='center',fontsize=8,color='#2c3e50')
    # despegue con velocidad
    ax.annotate('',xy=(6,1.8),xytext=(3.6,1.1),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2.5))
    ax.text(5.2,1.9,'v despegue = 9,00 m/s',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax.text(4,0.1,'v₀=0 → acelera en 1,20 m → sale a 9,00 m/s',ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0,7); ax.set_ylim(0,2.3); ax.axis('off')
    ax.set_title('Lanzamiento del tejo: aceleración en distancia fija')
    plt.tight_layout(); plt.show()
esquema()

def tejo(v_despegue=9.00, d=1.20):
    a=v_despegue**2/(2*d)
    t=v_despegue/a if a>0 else 0
    # con error: a=27 (o el que el usuario quiera comparar)
    x=np.linspace(0,d,100)
    v=np.sqrt(2*a*x)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4))
    ax1.plot(x,v,color='#2980b9',lw=2.5)
    ax1.scatter([d],[v_despegue],color='#27ae60',s=70,zorder=5,label=f'despegue: {v_despegue:.3g} m/s')
    ax1.set_xlabel('distancia recorrida (m)'); ax1.set_ylabel('velocidad (m/s)')
    ax1.set_title(f'v crece con la distancia (a={a:.3g} m/s²)'); ax1.legend(fontsize=8); ax1.grid(alpha=.3)
    # comparar velocidad de salida vs aceleración aplicada
    accs=np.linspace(a*0.6,a*1.2,50)
    vs=np.sqrt(2*accs*d)
    ax2.plot(accs,vs,color='#8e44ad',lw=2.5)
    ax2.scatter([a],[v_despegue],color='#27ae60',s=70,zorder=5,label='requerida')
    ax2.scatter([27],[np.sqrt(2*27*d)],color='#c0392b',s=70,zorder=5,label=f'con error (27): {np.sqrt(2*27*d):.3g} m/s')
    ax2.set_xlabel('aceleración de impulso (m/s²)'); ax2.set_ylabel('velocidad de salida (m/s)')
    ax2.set_title('Menos aceleración → menos velocidad de salida'); ax2.legend(fontsize=8); ax2.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(tejo,
    v_despegue=FloatSlider(value=9.00,min=5,max=14,step=1,description='v salida (m/s)'),
    d=FloatSlider(value=1.20,min=0.8,max=2.0,step=0.2,description='d impulso (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                ee=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+ee
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    d=1.20; vd=9.00
    a=vd**2/(2*d); t=vd/a
    _pc(f"a) ACELERACIÓN (usando v²=v₀²+2ad con v₀=0):")
    _pc(f"   a = v²/(2d) = 9,00²/(2·1,20) = {a:.3g} m/s²")
    _pc("")
    _pc(f"b) TIEMPO de la fase de impulso (usando v=v₀+at):")
    _pc(f"   t = v/a = 9,00/{a:.3g} = {t:.3g} s")
    _pc("")
    import numpy as _np
    v_err=_np.sqrt(2*27.0*d)
    _pc(f"c) Con aceleración reducida a 27,0 m/s² (20% menor):")
    _pc(f"   v = √(2·a·d) = √(2·27,0·1,20) = {v_err:.3g} m/s")
    _pc(f"   La velocidad baja de 9,00 a {v_err:.3g} m/s: una reducción del")
    _pc(f"   {(1-v_err/9.0)*100:.1f}% en velocidad, no del 20%. La relación NO es lineal:")
    _pc("   como v∝√a, un 20% menos de aceleración da solo ~10,6% menos de velocidad.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("El tejo parte del reposo y sale a 9,00 m/s tras acelerar en d=1,20 m. Calcule la aceleración (en m/s²).",33.8,0.03,"m/s²",
    "a=v²/(2d)=9,00²/(2·1,20)=33,8 m/s².", id_preg="Cin_p3_1", peso=2.0)
quiz_numerico("¿Cuánto dura la fase de impulso del tejo? (en s)",0.267,0.05,"s",
    "t=v/a=9,00/33,8≈0,267 s.", id_preg="Cin_p3_2", peso=2.0)
quiz_numerico("Si por error la aceleración es solo 27,0 m/s² (sobre la misma distancia 1,20 m), ¿cuál es la nueva velocidad de salida? (en m/s)",8.05,0.03,"m/s",
    "v=√(2·a·d)=√(2·27,0·1,20)≈8,05 m/s.", id_preg="Cin_p3_3", peso=2.0)
quiz_opcion_multiple("Una reducción del 20% en la aceleración produce una reducción de velocidad de salida de:",
    ["Exactamente 20%","Alrededor de 10,6%, porque v∝√a (relación no lineal)","40%","Ninguna, la velocidad no cambia"],1,
    "Como v=√(2ad), la velocidad es proporcional a √a. Reducir a en 20% baja v solo ~10,6%.", id_preg="Cin_p3_4", peso=2.0)


---
## 📊 Tu calificación del taller
Pulsa el botón para calcular tu nota formativa (0,0 a 5,0), ponderada por la dificultad de cada punto. Puedes reintentar las preguntas las veces que quieras: la práctica es para aprender.


In [ ]:
calificacion_final(total_preguntas=13)

---
### ✅ Fin del taller
Has trabajado la conversión de unidades imperiales a SI, el análisis de gráficas v-t (aceleración como pendiente, desplazamiento como área), la diferencia entre desplazamiento y distancia, y la cinemática de aceleración constante aplicada al tejo. Estos son los cimientos de toda la mecánica.

© 2026 Juan David Betancur Ríos, docente de física para ingeniería. Material educativo de uso académico (Física BR). Todos los derechos reservados. Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico del autor.
